# Visualization
# Author: Yangshunshun Lu

**1. What was done:**
This notebook transforms the high-dimensional NLP and audio features into intuitive, interpretable visual narratives. It executes a comprehensive suite of plots, including histograms for lexical richness, correlation heatmaps for pure NLP metrics, and Word Clouds weighted by TF-IDF scores (both globally and topic-specific). Furthermore, it tracks the temporal evolution of LDA topics and keywords using time-series area and line charts. Advanced techniques such as Z-score anomaly detection and 2D PCA (Principal Component Analysis) clustering were also deployed to map the semantic space, culminating in a fully interactive `pyLDAvis` web dashboard.

**2. Why it was done:**
While algorithms like LDA and TF-IDF successfully quantify text, their raw mathematical outputs (probabilities and sparse matrices) are inherently abstract. The primary objective of this visualization pipeline is to visually validate the feature engineering process and uncover hidden macro-patterns. By plotting temporal trends and semantic clusters, we bridge the gap between machine learning outputs and human-readable cultural insights, providing a vital exploratory layer before diving into the final statistical regressions.

**3. Key Findings & Results:**
* **Thematic Evolution over Time:** The time-series area charts successfully mapped the life cycle of the 5 LDA topics, revealing the dynamic fluctuations of dominant cultural themes in pop music across different years and months.
* **Semantic Clustering & Anomaly Detection:** The 2D PCA projection effectively clustered the tracks based on their TF-IDF vectors, visually corroborating the distinctness of our LDA topics. Concurrently, Z-score scatter plots successfully pinpointed extreme lyrical outliers—tracks with exceptionally high word counts or lexical sparsity.
* **Interactive Topic Dashboard:** The deployment of the `pyLDAvis` module generated a dynamic HTML dashboard (`LDA_Interactive_Dashboard.html`), allowing for an interactive deep-dive into the marginal topic distributions and the most salient terms defining each musical cluster.

**4. AI Disclosure:**
Gemini used in learning LDAvis and sklearn's TfidfVectorizer, CountVectorizer and PCA, since they are not taught in course but we use these methods for visualization completeness and generate more insights in the data.

## Set up and Data Loading

In [ ]:
# ==========================================
# Visualization Setup & Global Data Loading
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from wordcloud import WordCloud
from scipy.stats import zscore
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import PCA, LatentDirichletAllocation

# Set global plotting theme
sns.set_theme(style="whitegrid", palette="muted")

# 1. Define matching file paths
DATA_DIR = os.path.join('..', '..', 'Data', 'cleaned_data')
model_file = os.path.join(DATA_DIR, 'lyricsFeatures_mergeERUS_clean_enonly_token_pos_ldaTfidf.csv')
global_tfidf_file = os.path.join(DATA_DIR, 'global_tfidf_ranking.csv')

# 2. Load data ONCE for all subsequent cells
print("Loading main datasets...")
df = pd.read_csv(model_file)
df_global = pd.read_csv(global_tfidf_file)

# 3. Global Preprocessing
df['pos_filtered_lyrics'] = df['pos_filtered_lyrics'].fillna('')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

print(f"✅ Data loaded successfully! Total tracks: {len(df)}")

Loading main datasets...
✅ Data loaded successfully! Total tracks: 22030


## NLP Features Distribution (Histograms & Boxplots)

In [2]:
# ==========================================
# 1. Text Length & Lexical Richness Distributions
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '1_text_length')
os.makedirs(out_dir, exist_ok=True)

# 1A: Word Count Histogram
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='word_count', bins=40, kde=True, color='#4CB391')
plt.title('Distribution of Lyrics Word Count', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Word Count (Total Words)', fontsize=12)
plt.ylabel('Number of Tracks', fontsize=12)
plt.axvline(df['word_count'].median(), color='red', linestyle='--', label=f"Median: {df['word_count'].median():.0f}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '1A_WordCount_Histogram.png'), dpi=300)
plt.close()

# 1B: Lexical Richness Histogram
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='lexical_richness', bins=40, kde=True, color='#8993F8')
plt.title('Distribution of Lexical Richness', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Lexical Richness (Unique / Total Words)', fontsize=12)
plt.ylabel('Number of Tracks', fontsize=12)
plt.axvline(df['lexical_richness'].median(), color='red', linestyle='--', label=f"Median: {df['lexical_richness'].median():.2f}")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '1B_LexicalRichness_Histogram.png'), dpi=300)
plt.close()

# 1C: Topic vs Word Count
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Dominant_Topic', y='word_count', hue='Dominant_Topic', palette='Set2', legend=False)
sns.stripplot(data=df, x='Dominant_Topic', y='word_count', color=".25", alpha=0.3, jitter=True)
plt.title('Word Count Distribution across LDA Topics', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '1C_Topic_vs_WordCount_Boxplot.png'), dpi=300)
plt.close()

# 1D: Topic vs Lexical Richness
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Dominant_Topic', y='lexical_richness', hue='Dominant_Topic', palette='Set3', legend=False)
sns.stripplot(data=df, x='Dominant_Topic', y='lexical_richness', color=".25", alpha=0.3, jitter=True)
plt.title('Lexical Richness across LDA Topics', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '1D_Topic_vs_Richness_Boxplot.png'), dpi=300)
plt.close()

print("✅ Section 1 plots generated.")

✅ Section 1 plots generated.


## Correlation Heatmap

In [3]:
# ==========================================
# 2. Correlation Analysis
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '2_correlation')
os.makedirs(out_dir, exist_ok=True)

pure_nlp_features = [
    'word_count', 'lexical_richness', 'avg_word_length', 'text_sentiment',
    'Topic_0_Prob', 'Topic_1_Prob', 'Topic_2_Prob', 'Topic_3_Prob', 'Topic_4_Prob'
]

corr_matrix = df[pure_nlp_features].corr(method='spearman')

plt.figure(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="PRGn", center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title('Correlation Heatmap of Pure NLP Features', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.savefig(os.path.join(out_dir, '2A_NLP_Correlation_Heatmap.png'), dpi=300)
plt.close()
print("✅ Section 2 plots generated.")

✅ Section 2 plots generated.


## TF-IDF Rankings & Word Clouds

In [4]:
# ==========================================
# 3. TF-IDF & Word Clouds
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '3_tfidf')
os.makedirs(out_dir, exist_ok=True)

# 3A: Global Top 20 Words Bar Chart
top_20_words = df_global.head(20)
plt.figure(figsize=(12, 8))
sns.barplot(data=top_20_words, x='TF-IDF_score', y='term', hue='term', palette='viridis', legend=False)
plt.title('Top 20 Global Keywords by TF-IDF Score', fontsize=18, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '3A_Top20_TFIDF_BarChart.png'), dpi=300)
plt.close()

# 3B: Global Word Cloud
tfidf_dict = dict(zip(df_global['term'], df_global['TF-IDF_score']))
wc_global = WordCloud(width=1200, height=800, background_color='white', 
                      colormap='magma', max_words=150).generate_from_frequencies(tfidf_dict)
plt.figure(figsize=(12, 8))
plt.imshow(wc_global, interpolation='bilinear')
plt.axis('off') 
plt.title('Global Lyrics Word Cloud (Sized by TF-IDF)', fontsize=18, fontweight='bold', pad=20)
plt.savefig(os.path.join(out_dir, '3B_Global_WordCloud.png'), dpi=300)
plt.close()

# 3C: Topic Specific Word Clouds
colormaps = ['Blues', 'Reds', 'Greens', 'Purples', 'Oranges']
for topic_idx in sorted(df['Dominant_Topic'].unique()):
    topic_lyrics = " ".join(df[df['Dominant_Topic'] == topic_idx]['pos_filtered_lyrics'].tolist())
    if not topic_lyrics.strip(): continue

    wc_topic = WordCloud(width=800, height=600, background_color='white', 
                         colormap=colormaps[topic_idx % len(colormaps)],
                         max_words=100, collocations=False).generate(topic_lyrics)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(wc_topic, interpolation='bilinear')
    plt.axis('off')
    plt.title(f'Word Cloud for LDA Topic {topic_idx}', fontsize=16, fontweight='bold', pad=15)
    plt.savefig(os.path.join(out_dir, f'3_Topic_{topic_idx}_WordCloud.png'), dpi=300)
    plt.close()

print("✅ Section 3 plots generated.")

✅ Section 3 plots generated.


## Time Series Evolution

In [5]:
# ==========================================
# 4. Time Series Evolution
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '4_timeseries')
os.makedirs(out_dir, exist_ok=True)

# Create a temporary dataframe dropping NaT dates to avoid messing up global df
ts_df = df.dropna(subset=['date']).copy()
ts_df['year_month'] = ts_df['date'].dt.to_period('M')

# 4A: Topic Evolution Area Chart
topic_cols = [f'Topic_{i}_Prob' for i in range(5)]
monthly_topics = ts_df.groupby('year_month')[topic_cols].mean()
monthly_topics.index = monthly_topics.index.to_timestamp()

plt.figure(figsize=(12, 6))
monthly_topics.plot(kind='area', stacked=True, alpha=0.75, colormap='Set3', figsize=(12, 6))
plt.title('Evolution of LDA Topics Over Time', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Average Topic Probability')
plt.legend(title='LDA Topics', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '4A_Topic_Evolution_AreaChart.png'), dpi=300)
plt.close()

# 4B: Keyword Trend Line Chart
top_3_words = df_global['term'].head(3).tolist()
for word in top_3_words:
    ts_df[f'contains_{word}'] = ts_df['pos_filtered_lyrics'].str.contains(rf'\b{word}\b', regex=True).astype(int)

monthly_words = ts_df.groupby('year_month')[[f'contains_{w}' for w in top_3_words]].mean()
monthly_words.index = monthly_words.index.to_timestamp()

plt.figure(figsize=(12, 6))
markers, colors = ['o', 's', '^'], ['#e63946', '#457b9d', '#2a9d8f']
for i, word in enumerate(top_3_words):
    plt.plot(monthly_words.index, monthly_words[f'contains_{word}'], 
             marker=markers[i], color=colors[i], linewidth=2.5, markersize=8, label=f'Keyword: "{word}"')

plt.title('Top 3 Global Keywords Popularity Over Time', fontsize=16, fontweight='bold', pad=15)
plt.ylabel('Proportion of Tracks Containing Keyword')
plt.legend(title='Keywords')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '4B_Keyword_Trend_LineChart.png'), dpi=300)
plt.close()

print("✅ Section 4 plots generated.")

✅ Section 4 plots generated.


<Figure size 1200x600 with 0 Axes>

## Anomalies & PCA

In [6]:
# ==========================================
# 5. Anomalies & PCA Clustering
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '5_anomalies')
os.makedirs(out_dir, exist_ok=True)

# 5A: Z-Score Outlier Detection
df['z_word_count'] = zscore(df['word_count'])
df['z_lexical_richness'] = zscore(df['lexical_richness'])
df['is_outlier'] = (df['z_word_count'].abs() > 2.5) | (df['z_lexical_richness'].abs() > 2.5)

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df[~df['is_outlier']], x='z_word_count', y='z_lexical_richness', color='lightgrey', alpha=0.6, label='Normal Tracks')
sns.scatterplot(data=df[df['is_outlier']], x='z_word_count', y='z_lexical_richness', color='red', edgecolor='black', s=80, label='Anomalies')

extreme_outliers = df[df['is_outlier']].copy()
extreme_outliers['total_dev'] = extreme_outliers['z_word_count'].abs() + extreme_outliers['z_lexical_richness'].abs()
for _, row in extreme_outliers.sort_values('total_dev', ascending=False).head(5).iterrows():
    plt.text(row['z_word_count']+0.1, row['z_lexical_richness']+0.1, str(row['track_name'])[:15]+'...', fontsize=9, color='darkred')

plt.axvline(0, color='blue', linestyle='--', alpha=0.3)
plt.axhline(0, color='blue', linestyle='--', alpha=0.3)
plt.title('Z-Score Outlier Detection: Word Count vs. Lexical Richness', fontsize=16, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '5A_ZScore_Outliers_Scatter.png'), dpi=300)
plt.close()

# 5B: TF-IDF PCA Clustering
tfidf_vectorizer = TfidfVectorizer(max_df=0.9, min_df=5)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['pos_filtered_lyrics']).toarray()
pca_result = PCA(n_components=2, random_state=42).fit_transform(tfidf_matrix)

df['pca_x'], df['pca_y'] = pca_result[:, 0], pca_result[:, 1]
plt.figure(figsize=(11, 8))
sns.scatterplot(data=df, x='pca_x', y='pca_y', hue='Dominant_Topic', palette='Set1', alpha=0.7, s=50, edgecolor='white')
plt.title('2D PCA Projection of TF-IDF Space (Colored by LDA Topic)', fontsize=16, fontweight='bold')
plt.legend(title='LDA Topic', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '5B_TFIDF_PCA_Clustering.png'), dpi=300)
plt.close()

print("✅ Section 5 plots generated.")

✅ Section 5 plots generated.


## LDA Topic Insights

In [7]:
# ==========================================
# 6. LDA Topic Distribution & Insights
# ==========================================
out_dir = os.path.join('..', 'result', 'figure', '6_lda_topics')
os.makedirs(out_dir, exist_ok=True)

# 6A: Dominant Topic Pie Chart
topic_counts = df['Dominant_Topic'].value_counts().sort_index()
colors = sns.color_palette("pastel")[0:len(topic_counts)]
explode = [0.05 if i == topic_counts.idxmax() else 0 for i in range(len(topic_counts))]

plt.figure(figsize=(10, 8))
plt.pie(topic_counts, labels=[f"Topic {i}" for i in topic_counts.index], colors=colors, 
        explode=explode, autopct='%1.1f%%', startangle=140, 
        textprops={'fontsize': 14, 'fontweight': 'bold'}, shadow=True)
plt.title('Distribution of Dominant LDA Topics', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(out_dir, '6A_Dominant_Topic_PieChart.png'), dpi=300)
plt.close()

# 6B: Top 10 Words per Topic
colormaps = ['Blues_r', 'Reds_r', 'Greens_r', 'Purples_r', 'Oranges_r']
for topic_idx in sorted(df['Dominant_Topic'].unique()):
    topic_docs = df[df['Dominant_Topic'] == topic_idx]['pos_filtered_lyrics'].tolist()
    if not topic_docs: continue
        
    vectorizer = CountVectorizer(max_df=0.9, min_df=2)
    tf_matrix = vectorizer.fit_transform(topic_docs)
    
    topic_df = pd.DataFrame({'word': vectorizer.get_feature_names_out(), 'count': tf_matrix.sum(axis=0).A1})
    top_10 = topic_df.sort_values(by='count', ascending=False).head(10)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=top_10, x='count', y='word', hue='word', palette=colormaps[topic_idx % len(colormaps)], legend=False)
    plt.title(f'Top 10 Representative Words for LDA Topic {topic_idx}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'6B_Topic_{topic_idx}_TopWords_Bar.png'), dpi=300)
    plt.close()

print("✅ Section 6 plots generated.")

✅ Section 6 plots generated.


## Interactive pyLDAvis

In [8]:
# ==========================================
# 7. Interactive pyLDAvis Dashboard
# ==========================================
import pyLDAvis

out_dir = os.path.join('..', 'result', 'figure')

# Note: For pyLDAvis to work, we must rebuild the LDA model exactly once here
docs = df['pos_filtered_lyrics'].tolist()
tf_vectorizer = CountVectorizer(max_df=0.9, min_df=5)
tf_matrix = tf_vectorizer.fit_transform(docs)

lda_model = LatentDirichletAllocation(n_components=5, max_iter=10, random_state=42)
lda_model.fit(tf_matrix)

try:
    import pyLDAvis.lda_model
    vis_data = pyLDAvis.lda_model.prepare(lda_model, tf_matrix, tf_vectorizer)
except ImportError:
    import pyLDAvis.sklearn
    vis_data = pyLDAvis.sklearn.prepare(lda_model, tf_matrix, tf_vectorizer)

output_path = os.path.join(out_dir, "LDA_Interactive_Dashboard.html")
pyLDAvis.save_html(vis_data, output_path)

print(f"✨ All visualizations successfully generated! Interactive dashboard saved to: {output_path}")

✨ All visualizations successfully generated! Interactive dashboard saved to: ..\result\figure\LDA_Interactive_Dashboard.html
